In [23]:
# Imports
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader, Dataset
from PIL import Image
import os
import numpy as np
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report, roc_auc_score

# Device detection (CPU fallback if no CUDA)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)




Using device: cpu


In [24]:
# Data Augmentation
ssl_transform = transforms.Compose([
    transforms.RandomResizedCrop(size=224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([transforms.ColorJitter(0.4,0.4,0.4,0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.GaussianBlur(3),
    transforms.ToTensor()
])

# Simple transform for evaluation (no augmentation)
eval_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])


In [25]:
# Dataset 
class SSLDataset(torch.utils.data.Dataset):
    def __init__(self, root, transform):
        self.dataset = datasets.ImageFolder(root=root, transform=None)
        self.transform = transform

    def __getitem__(self, idx):
        path, label = self.dataset.samples[idx]
        image = self.dataset.loader(path)
        # Two augmented versions of the same image
        x1 = self.transform(image)
        x2 = self.transform(image)
        return x1, x2, label

    def __len__(self):
        return len(self.dataset)


In [26]:
# Loading Train Data
train_data = SSLDataset(root="data/train", transform=ssl_transform)
train_loader = DataLoader(train_data, batch_size=64, shuffle=True, drop_last=True)

# For evaluation (labels needed)
eval_data = datasets.ImageFolder(root="data/train", transform=eval_transform)
eval_loader = DataLoader(eval_data, batch_size=64, shuffle=False)
class_names = eval_data.classes
print("Classes:", class_names)


Classes: ['fake', 'real']


In [27]:
# SimCLR Model
class SimCLR(nn.Module):
    def __init__(self, base_encoder, projection_dim=128):
        super(SimCLR, self).__init__()
        self.encoder = base_encoder
        dim_mlp = self.encoder.fc.in_features

        # Replace final FC with identity
        self.encoder.fc = nn.Identity()

        # Projection head
        self.projector = nn.Sequential(
            nn.Linear(dim_mlp, dim_mlp),
            nn.ReLU(),
            nn.Linear(dim_mlp, projection_dim)
        )

    def forward(self, x):
        h = self.encoder(x)          # Feature
        z = self.projector(h)        # Projection
        return h, z


In [28]:
# Contrastive Loss (NT-Xent)
def nt_xent_loss(z1, z2, temperature=0.5):
    batch_size = z1.shape[0]
    z1 = F.normalize(z1, dim=1)
    z2 = F.normalize(z2, dim=1)

    representations = torch.cat([z1, z2], dim=0)
    similarity_matrix = torch.matmul(representations, representations.T)

    # Mask self-comparisons
    mask = torch.eye(2*batch_size, dtype=torch.bool, device=z1.device)
    similarity_matrix = similarity_matrix[~mask].view(2*batch_size, -1)

    positives = torch.cat([torch.sum(z1 * z2, dim=-1),
                           torch.sum(z2 * z1, dim=-1)], dim=0)

    positives = positives.unsqueeze(1)
    logits = torch.cat([positives, similarity_matrix], dim=1)
    labels = torch.zeros(2*batch_size, dtype=torch.long, device=z1.device)

    logits /= temperature
    loss = F.cross_entropy(logits, labels)
    return loss


In [29]:
# SSL Training Loop
def train_ssl(model, loader, epochs=10, lr=3e-4, device="cuda"):
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for x1, x2, _ in loader:
            x1, x2 = x1.to(device), x2.to(device)
            _, z1 = model(x1)
            _, z2 = model(x2)

            loss = nt_xent_loss(z1, z2)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {total_loss/len(loader):.4f}")

    torch.save(model.state_dict(), "simclr_encoder.pth")
    return model


In [30]:
# Linear Evaluation 
def linear_evaluation(model, eval_loader, device="cuda"):
    model = model.to(device)
    model.eval()

    features, labels = [], []
    with torch.no_grad():
        for images, targets in eval_loader:
            images = images.to(device)
            h, _ = model(images)  # only encoder output
            features.append(h.cpu().numpy())
            labels.append(targets.numpy())

    features = np.concatenate(features)
    labels = np.concatenate(labels)

    # Train logistic regression classifier
    clf = LogisticRegression(max_iter=500).fit(features, labels)
    preds = clf.predict(features)

    acc = accuracy_score(labels, preds)
    prec = precision_score(labels, preds, average="binary")
    rec = recall_score(labels, preds, average="binary")
    f1 = f1_score(labels, preds, average="binary")
    cm = confusion_matrix(labels, preds)

    print(f"Accuracy:  {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall:    {rec:.4f}")
    print(f"F1-score:  {f1:.4f}")
    print("Confusion Matrix:\n", cm)


In [32]:
# Base encoder = ResNet18
base_encoder = models.resnet18(weights=None)
simclr = SimCLR(base_encoder)

# Train SSL (unsupervised)
simclr = train_ssl(simclr, train_loader, epochs=10, device="cpu")

# Evaluate with linear classifier (supervised)
linear_evaluation(simclr, eval_loader, device="cpu")


Epoch [1/10], Loss: 4.6216
Epoch [2/10], Loss: 4.4393
Epoch [3/10], Loss: 4.3500
Epoch [4/10], Loss: 4.2762
Epoch [5/10], Loss: 4.1970
Epoch [6/10], Loss: 4.1413
Epoch [7/10], Loss: 4.1322
Epoch [8/10], Loss: 4.0774
Epoch [9/10], Loss: 4.0876
Epoch [10/10], Loss: 4.0613
Accuracy:  0.9339
Precision: 0.9358
Recall:    0.9654
F1-score:  0.9504
Confusion Matrix:
 [[451  65]
 [ 34 948]]


In [3]:

# Step 6: Prediction on All Images in the 'check' folder
def predict_image(model, img_path):
    model.eval()  # Set model to evaluation mode
    img = Image.open(img_path).convert('RGB')
    img = transform(img).unsqueeze(0)  # Add batch dimension
    
    with torch.no_grad():
        output = model(img)
        _, predicted = torch.max(output, 1)
    
    # If labels were swapped (i.e., real=1, fake=0), we reverse the mapping here:
    return 'fake' if predicted.item() == 1 else 'real'

# Step 7: Show Image with Prediction
def show_image_with_prediction(img_path, prediction):
    img = Image.open(img_path)
    plt.imshow(img)
    plt.title(f"Prediction: {prediction}")
    plt.axis('off')
    plt.show()


# Step 8: Create Path File to Log Predictions
def create_path_file_and_predict(check_folder, model):
    # Create or open the path file to save the predictions
    log_file_path = "prediction_paths.txt"
    with open(log_file_path, "w") as log_file:
        log_file.write("Image Path, Prediction\n")
        
        # Loop through all files in the check folder and predict if they are real or fake
        for img_name in os.listdir(check_folder):
            img_path = os.path.join(check_folder, img_name)
            
            # Check if the file is an image (jpg, jpeg, or png)
            if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                result = predict_image(model, img_path)
                log_file.write(f"{img_path}, {result}\n")
                print(f"Prediction for {img_name}: {result}")
                
                # Visualize the result for the current image
                show_image_with_prediction(img_path, result)

check_folder = 'data/check'
create_path_file_and_predict(check_folder, simclr)


NameError: name 'simclr' is not defined